# Olive Fly Detection using Neural Network

This notebook implements an olive fly detection system using a lightweight neural network.
It includes both training and classification pipelines for detecting olive flies in olive fruit images.

# Introduction

The main goal of this project is to develop a software system capable of detecting olive flies in digital images. To achieve this, a lightweight machine learning model was implemented and evaluated. Lightweight models are particularly advantageous because they require relatively little computational power and memory, making them suitable for deployment on embedded and edge devices. In addition, their low computational complexity enables fast inference, allowing images to be processed efficiently in real-time application.


## 1. Import Required Libraries

In [98]:
import os
import glob
import cv2
import numpy as np
from skimage.measure import label
from sklearn.neural_network import MLPClassifier

## 2. Image Processing Functions

Before feature extraction can be performed, irrelevant information such as the image background must be separated from the insect. This allows the model to focus only on the object of interest. The preprocessing pipeline consists of several steps. First, the image is converted to grayscale. Next, Otsu thresholding is applied to distinguish the foreground from the background. Morphological closing is then used to fill small gaps and reduce noise in the resulting binary image. Finally, connected-component labeling is performed, and the largest connected region is selected as the insect mask. This mask is subsequently used for feature extraction.

In [99]:
def extract_foreground_mask(img, kernel_size=9):
    """Extract foreground (olive fruit) from the image."""
    img_gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY).astype(np.uint8)
    _, img_bw = cv2.threshold(img_gray, -1, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    
    kernel = np.ones((kernel_size, kernel_size))
    img_bw_cleaned = cv2.morphologyEx(img_bw, cv2.MORPH_CLOSE, kernel)
    
    labels = label(img_bw_cleaned)
    flat_labels = labels.flat
    flat_bw = img_bw_cleaned.flat
    
    if len(flat_bw) == 0 or np.sum(flat_bw) == 0:
        return np.zeros_like(img_gray)
        
    label_of_largest_region = np.argmax(np.bincount(flat_labels, weights=flat_bw))
    return (labels == label_of_largest_region).astype(np.uint8)

## 3. Features Used for Classification

| Feature | Description | Purpose |
|----------|-------------|----------|
| **Area** | Number of pixels in the foreground mask | Estimates insect size |
| **Aspect Ratio** | Width divided by height of the bounding box | Captures body shape |
| **Mean Saturation** | Average saturation value in HSV color space | Captures color characteristics |

### 3.1 Feature Justification

#### 3.1.1 Area
The **area** is calculated as the number of pixels belonging to the insect in the foreground mask. This feature provides an estimate of the insect's size within the image. Olive flies generally fall within a specific size range, and therefore area can help distinguish them from significantly larger or smaller insects. While image scale and distance from the camera can affect the measured area, it still provides useful information when combined with other features.

#### 3.1.2 Aspect Ratio
The **aspect ratio**  is computed as the width divided by the height of the insect's bounding box. This feature captures the overall shape of the insect. Different insect species often exhibit distinct body proportions; some are more elongated while others appear more compact. By incorporating aspect ratio, the model can use geometric characteristics of the detected object to aid classification.

#### 3.1.3 Mean Saturation
The **mean saturation** is calculated from the saturation channel of the HSV color representation. Unlike grayscale intensity, saturation measures the strength or purity of colors present in the image. Olive flies possess characteristic coloration that may differ from other insects in the dataset. The average saturation therefore provides a simple way of capturing color information, allowing the model to exploit appearance differences that are not reflected in shape or size features.

### 3.1.4 Summary

Individually, each feature provides only limited information about the insect. However, when used together they describe three complementary characteristics: size (area), shape (aspect ratio), and color (mean saturation). This combination allows the neural network to make classification decisions based on multiple aspects of the insect's appearance, improving its ability to distinguish olive flies from non-olive-fly specimens.

In [100]:
def compute_features(image):
    """Compute feature vector from image."""
    mask = extract_foreground_mask(image)
    area = float(np.sum(mask))
    
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if contours:
        c = max(contours, key=cv2.contourArea)
        x, y, w, h = cv2.boundingRect(c)
        aspect_ratio = float(w) / float(h) if h > 0 else 1.0
    else:
        aspect_ratio = 1.0
        
    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
    saturation_channel = hsv[:, :, 1]
    mean_saturation = float(cv2.mean(saturation_channel, mask=mask)[0])
    
    return np.array([area, aspect_ratio, mean_saturation])

## Training Phase

Load training data from the `olive_fly/` and `not_olive_fly/` directories and train the neural network classifier.

In [101]:
# Prepare training data
X_list, y_list = [], []

print("Reading training data...")

# Load negative samples (no olive fly)
for f in (glob.glob("not_olive_fly/*.JPG") + glob.glob("not_olive_fly/*.jpg")):
    img = cv2.imread(f)
    if img is not None:
        X_list.append(compute_features(img))
        y_list.append(0)
            
# Load positive samples (contains olive fly)
for f in (glob.glob("olive_fly/*.JPG") + glob.glob("olive_fly/*.jpg")):
    img = cv2.imread(f)
    if img is not None:
        X_list.append(compute_features(img))
        y_list.append(1)

print(f"Loaded {len(X_list)} training samples")
print(f"Positive samples (olive fly): {sum(y_list)}")
print(f"Negative samples (no olive fly): {len(y_list) - sum(y_list)}")

Reading training data...
Loaded 4700 training samples
Positive samples (olive fly): 630
Negative samples (no olive fly): 4070


## 4. Feature Normalization

The three extracted features differ significantly in scale. The area of the detected insect can reach several thousand pixels, whereas the aspect ratio is typically close to one and the mean saturation falls within a much smaller numerical range. Using these values directly would cause the larger features to have a disproportionate effect on the neural network calculations.
To avoid this issue, the features are standardized before being passed to the network:
[
x_{scaled} = \frac{x - \mu}{\sigma}
]
where (\mu) is the mean value of the feature and (\sigma) is its standard deviation. The values stored in X_MEAN and X_STD were calculated from the dataset and are used during inference to transform each feature.
After normalization, all features have a comparable scale, allowing the network to make use of information from size, shape and colour without one feature dominating the others.


In [ ]:
# Convert to numpy arrays
X, y = np.array(X_list), np.array(y_list)

# Feature Standardization Scaling
X_mean, X_std = X.mean(axis=0), X.std(axis=0)
X_std[X_std == 0] = 1e-8  # Avoid division by zero
X_scaled = (X - X_mean) / X_std

print("Feature statistics:")
print(f"X_mean: {X_mean}")
print(f"X_std: {X_std}")

# Train Neural Network: 3 inputs -> 4 hidden neurons -> 1 output
mlp = MLPClassifier(hidden_layer_sizes=(4,), activation='relu', max_iter=5000, random_state=42)
mlp.fit(X_scaled, y)

print("\nNeural Network Training Complete!")

Feature statistics:
X_mean: [3.87745872e+03 1.11746390e+00 1.49344926e+02]
X_std: [4.80163557e+03 5.52435110e-01 5.18820731e+01]

Neural Network Training Complete!


### Extract and Display Model Parameters

Copy these parameters for use in the classification phase:

In [103]:
print(f"X_MEAN = np.array({list(X_mean)})")
print(f"X_STD  = np.array({list(X_std)})")
print(f"W1 = np.array({mlp.coefs_[0].tolist()})")
print(f"B1 = np.array({mlp.intercepts_[0].tolist()})")
print(f"W2 = np.array({mlp.coefs_[1].tolist()})")
print(f"B2 = np.array({mlp.intercepts_[1].tolist()})")

# Store the parameters for later use
MODEL_PARAMS = {
    'X_MEAN': X_mean,
    'X_STD': X_std,
    'W1': mlp.coefs_[0],
    'B1': mlp.intercepts_[0],
    'W2': mlp.coefs_[1],
    'B2': mlp.intercepts_[1]
}

X_MEAN = np.array([np.float64(3877.458723404255), np.float64(1.1174638977603828), np.float64(149.3449262126882)])
X_STD  = np.array([np.float64(4801.635565207151), np.float64(0.5524351102376486), np.float64(51.88207312731143)])
W1 = np.array([[2.2980583705693416, 0.37722473689468405, 0.11316033119249452, 1.4048232478170037, -4.764950783798769], [-1.4322630864482022, -1.8409220190051063, 0.4893649830271649, 1.3368083520358682, 0.537566710476442], [-5.101909303998162, 1.0391446307113144, 1.2020398019772356, -2.393111127767018, 0.9103724509538029]])
B1 = np.array([-0.7622016112089042, -1.7389236076202965, -0.8959346443569431, -2.107529611726016, -3.0968748795029826])
W2 = np.array([[-2.377960475015611], [-1.1384342273650265], [-0.6113732434401838], [-1.5627550114450623], [-3.2570875219831823]])
B2 = np.array([0.5444275976098744])


## 5. Neural Network Architecture
The classifier is implemented as a small feedforward neural network. Since only three features are used as input, a simple architecture was considered sufficient for this task. The network receives the area, aspect ratio and mean saturation values and processes them through a hidden layer consisting of four neurons.
The first stage of the forward pass is computed as
[
z_1 = XW_1 + b_1
]
where (W_1) contains the hidden-layer weights and (b_1) contains the corresponding bias values.
A ReLU activation function is then applied:
[
a_1 = \max(0, z_1)
]
This operation replaces negative values with zero while keeping positive values unchanged. Introducing this non-linearity allows the network to learn relationships between the features that would not be possible using only linear transformations.
The hidden layer output is then passed to the final neuron:
[
z_2 = a_1W_2 + b_2
]
The result is converted into a probability using the sigmoid activation function:
[
p = \frac{1}{1 + e^{-z_2}}
]
The output value ranges between 0 and 1 and represents the confidence that the detected object is an olive fly. A threshold of 0.5 is used for classification. Values equal to or above this threshold are classified as olive flies, while lower values are classified as non-olive flies.
The network was intentionally kept small in order to minimise computational requirements while still being able to model non-linear relationships between the extracted features.


In [104]:
# Use the trained model parameters from above, or paste your own:
X_MEAN = MODEL_PARAMS['X_MEAN']
X_STD = MODEL_PARAMS['X_STD']

W1 = MODEL_PARAMS['W1']
B1 = MODEL_PARAMS['B1']

W2 = MODEL_PARAMS['W2']
B2 = MODEL_PARAMS['B2']

print("Model parameters loaded successfully!")

Model parameters loaded successfully!


In [ ]:
def detect_olive_fly(image) -> bool:

    try:
        # 1. Extract and standardize physical attributes
        raw_features = compute_features(image)
        scaled_features = (raw_features - X_MEAN) / X_STD
        
        # 2. Hidden Layer Propagation: Z1 = A1*W1 + B1
        z1 = np.dot(scaled_features, W1) + B1
        a2 = np.maximum(0, z1)  # ReLU non-linear activation function
        
        # 3. Output Layer Propagation: Z2 = A2*W2 + B2
        z2 = np.dot(a2, W2) + B2
        
        # Sigmoid activation function mapping to classification boundaries
        probability = 1 / (1 + np.exp(-np.clip(z2[0], -500, 500)))
        
        return bool(probability >= 0.5)
        
    except Exception as e:
        print(f"Error in detection: {e}")
        return False

### 6. Evaluation Method
The performance of the classifier is evaluated by comparing the predicted label with the known label of each image in the dataset. The results are grouped into four categories:
True Positives (TP): images containing an olive fly that are correctly classified.
True Negatives (TN): images not containing an olive fly that are correctly classified.
False Positives (FP): images that do not contain an olive fly but are incorrectly classified as one.
False Negatives (FN): images that contain an olive fly but are incorrectly classified as non-olive flies.
Using these values, precision and recall are calculated:
[
Precision = \frac{TP}{TP + FP}
]
[
Recall = \frac{TP}{TP + FN}
]
Precision indicates how reliable positive predictions are. For example, a high precision means that when the model identifies an olive fly, it is usually correct.
Recall measures how many of the actual olive flies are successfully detected. A high recall indicates that only a small number of olive flies are missed by the classifier.
Both metrics are important because they provide different information about the model's performance. A classifier may achieve high precision while missing many olive flies, or achieve high recall while producing many false detections. Considering both metrics gives a more complete assessment of the system.


In [106]:
def evaluate_model(test_directory, verbose=False):
    """
    Evaluate the model on test data.
    
    Args:
        test_directory: Path to directory containing test images
        verbose: Print detailed results for each image
    """
    TP, TN, FP, FN = 0, 0, 0, 0
    
    for filename in glob.glob(test_directory + "/**/*.JPG", recursive=True) + \
                    glob.glob(test_directory + "/**/*.jpg", recursive=True):
        img = cv2.imread(filename)
        if img is None:
            continue
            
        # Determine ground truth from directory name
        if "not_olive_fly" in filename:
            olive_fly = False
        elif "olive_fly" in filename:
            olive_fly = True
        else:
            print(f"Skipping {filename} - not labeled")
            continue

        detection_result = detect_olive_fly(img)

        # Update confusion matrix
        if olive_fly and detection_result:
            TP += 1
        elif olive_fly and not detection_result:
            FN += 1
        elif not olive_fly and detection_result:
            FP += 1
        else:
            TN += 1
            
        if verbose:
            result_str = "contains an olive fly" if detection_result else "does not contain an olive fly"
            print(f"{filename} {result_str}")
    
    print(f"\nConfusion Matrix:")
    print(f"TP: {TP}, TN: {TN}, FP: {FP}, FN: {FN}")
    
    if (TP + FP) > 0 and (TP + FN) > 0:
        precision = TP / (TP + FP)
        recall = TP / (TP + FN)
        f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
        print(f"\nMetrics:")
        print(f"Precision: {precision:.4f}")
        print(f"Recall: {recall:.4f}")
        print(f"F1-Score: {f1_score:.4f}")
    
    return TP, TN, FP, FN

In [107]:

print("Evaluating on training data...\n")
evaluate_model(".", verbose=False)


Evaluating on training data...


Confusion Matrix:
TP: 458, TN: 3094, FP: 976, FN: 172

Metrics:
Precision: 0.3194
Recall: 0.7270
F1-Score: 0.4438


(458, 3094, 976, 172)

## 7. Test Single Image
Use this section to test the detection on a single image:

In [108]:
test_image_path = "olive_fly/IMG_0597 29 referencia.JPG"

if os.path.exists(test_image_path):
    test_img = cv2.imread(test_image_path)
    if test_img is not None:
        result = detect_olive_fly(test_img)
        print(f"Image: {test_image_path}")
        print(f"Detection Result: {'Olive Fly Detected' if result else 'No Olive Fly'}")
    else:
        print(f"Could not read image: {test_image_path}")
else:
    print(f"File not found: {test_image_path}")

Image: olive_fly/IMG_0597 29 referencia.JPG
Detection Result: No Olive Fly


## 8. Conclusion

This project shows that olive fly detection can be effectively achieved using a lightweight machine learning approach based on simple image features and a small neural network. By extracting area, aspect ratio, and mean saturation, the system captures key information about the insect’s size, shape, and color while keeping the model computationally efficient.

The preprocessing pipeline ensures that features are extracted from a clean insect mask, improving reliability. Feature normalization further supports stable learning by placing all inputs on a comparable scale. The compact neural network is sufficient to model relationships between the features while remaining suitable for real-time or edge-device deployment.

Evaluation using precision and recall provides a balanced view of performance, highlighting both correct detections and missed cases. Overall, the results demonstrate that a simple, feature-based approach can be an effective solution for olive fly detection in resource-limited settings.